# সহজ ভাষায় Notebook Guide

এই notebook **সম্পূর্ণ self-contained** — project folder লাগে না। Monitoring-এর প্রতিটা ধারণা ছোট code দিয়ে মাপা হয়েছে: Ollama-র timing field, Prometheus bucket আর cardinality, OpenTelemetry GenAI span, সস্তা quality check, LLM judge, SLO burn rate, drift। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন। দরকার: `ollama`, `prometheus_client`, `opentelemetry-sdk`, `pandas`, `scipy`।
2. Section 1, 4, 5 real Ollama (`llama3.1:8b`) লাগে। বাকিগুলো Ollama ছাড়াই।
3. Prometheus server, Phoenix, Docker — কিছুই লাগে না।

# LLM Observability & Monitoring — Lecture Notebook

1. Ollama-র free telemetry: একটা call থেকে ছয়টা metric
2. Prometheus: 10 s-এর bucket ফাঁদ আর cardinality-র বিল
3. OpenTelemetry GenAI span: `chat llama3.1:8b`
4. সস্তা quality check — কী ধরে, কী ধরে না
5. LLM judge: দুই design, একই model
6. SLO burn rate: কখন page, কখন ticket
7. Drift: KS আর Wasserstein

In [1]:
import json, re, time, math
import numpy as np
import pandas as pd
import ollama

MODEL = "llama3.1:8b"
client = ollama.Client()
OPTS = {"temperature": 0, "seed": 42, "num_ctx": 8192}       # same options every call -> no model reloads
PASSAGES = [
    "Sick leave: employees get 12 paid sick days per year. Tell your manager before 10am on the first day.",
    "Annual leave: up to 5 unused days can be carried over to the next year; the rest expires on 31 March.",
    "Remote work: up to 3 days per week with manager approval. Core hours are 11:00 to 15:00 local time.",
]
CONTEXT = "\n".join(f"[{i}] {p}" for i, p in enumerate(PASSAGES, 1))
print("model present:", MODEL in [m.model for m in client.list().models])

model present: True


## 1. Ollama-র Free Telemetry

প্রতিটা Ollama response-এ: `prompt_eval_count`, `eval_count`, `done_reason`, আর nanosecond-এ `load_duration`, `prompt_eval_duration`, `eval_duration`, `total_duration`। কোনো library ছাড়াই golden signal-এর LLM অংশ বেরিয়ে আসে। Stream mode-এ client-side TTFT-ও আমরা নিজে মাপি।

In [2]:
def ask(question, nonce=None):
    prefix = nonce if nonce is not None else f"(nonce {time.time_ns()})\n"   # new nonce = never-seen prompt prefix
    t0 = time.perf_counter(); ttft = None; text = []
    for part in client.chat(model=MODEL, stream=True, options=OPTS, keep_alive="30m",
                            messages=[{"role": "system", "content": "Answer ONLY from the context and cite like [1]."},
                                      {"role": "user", "content": f"{prefix}CONTEXT:\n{CONTEXT}\n\nQUESTION: {question}"}]):
        if part.message.content and ttft is None:
            ttft = time.perf_counter() - t0
        text.append(part.message.content)
        last = part
    return "".join(text).strip(), last, ttft

answer, r, ttft = ask("How many sick days do employees get?")
ns = 1e9
telemetry = {
    "answer": answer,
    "prompt_tokens (in)": r.prompt_eval_count, "completion_tokens (out)": r.eval_count, "done_reason": r.done_reason,
    "load_ms (cold start?)": round(r.load_duration / 1e6),
    "prefill_ms": round(r.prompt_eval_duration / 1e6),
    "prefill tok/s": round(r.prompt_eval_count / (r.prompt_eval_duration / ns)),
    "decode tok/s": round(r.eval_count / (r.eval_duration / ns), 1),
    "server TTFT ≈ load+prefill (ms)": round((r.load_duration + r.prompt_eval_duration) / 1e6),
    "client TTFT (ms)": round(ttft * 1000),
    "GPU-seconds (cost proxy)": round((r.prompt_eval_duration + r.eval_duration) / ns, 2),
}
pd.Series(telemetry).to_frame("value")

,value
answer,12 [1]
prompt_tokens (in),138
completion_tokens (out),5
done_reason,stop
load_ms (cold start?),11
prefill_ms,1012
prefill tok/s,136
decode tok/s,23.8
server TTFT ≈ load+prefill (ms),1023
client TTFT (ms),1044


In [3]:
# one fresh prompt, sent twice: the second call reuses Ollama's cached prompt prefix
fresh = f"(nonce {time.time_ns()})\n"
_, first, _ = ask("How many sick days do employees get?", nonce=fresh)
_, again, t_again = ask("How many sick days do employees get?", nonce=fresh)
print(f"prefill: first {first.prompt_eval_duration/1e6:.0f} ms -> repeat {again.prompt_eval_duration/1e6:.0f} ms "
      f"(client TTFT {t_again*1000:.0f} ms)")

prefill: first 759 ms -> repeat 53 ms (client TTFT 57 ms)


**পড়ার নিয়ম:** `prompt_eval_duration` শুধু *uncached* token গোনে। একই prompt আবার এলে prefill প্রায় শূন্য — তাই TTFT-এর বণ্টন **দুই চূড়ার** (cache hit বনাম miss)। গড় দেখলে দুটোর কোনোটাই বোঝা যায় না; percentile দেখুন, আর load test-এ সবাই একই প্রশ্ন পাঠালে latency মিথ্যা রকম ভালো দেখায়।

## 2. Prometheus: Bucket ফাঁদ আর Cardinality

Histogram সংখ্যা রাখে না, রাখে **bucket-এর গণনা**; `histogram_quantile` bucket-এর ভেতরে linear interpolation করে percentile *অনুমান* করে। `prometheus_client`-এর default bucket 10 s-এ শেষ। নিচে পাঁচটা আসল-মাপের local LLM latency (২ থেকে ৪৫ s) দুই রকম bucket-এ, আর Prometheus-এর মতোই p95 হিসাব:

In [4]:
from prometheus_client import CollectorRegistry, Histogram, Counter, generate_latest

LLM_BUCKETS = (0.1, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 120)
latencies = [2.0, 8.0, 12.0, 30.0, 45.0]

def quantile_from_buckets(q, uppers, counts):
    # what histogram_quantile() does: find the bucket, interpolate linearly inside it
    total = counts[-1]; rank = q * total
    for i, (ub, c) in enumerate(zip(uppers, counts)):
        if c >= rank:
            if math.isinf(ub):
                return uppers[i - 1]          # Prometheus returns the highest finite bound
            lo, prev = (0.0, 0.0) if i == 0 else (uppers[i - 1], counts[i - 1])
            return lo + (ub - lo) * (rank - prev) / (c - prev)

rows = []
for name, buckets in [("default (ends at 10 s)", Histogram.DEFAULT_BUCKETS), ("LLM buckets (to 120 s)", LLM_BUCKETS)]:
    reg = CollectorRegistry()
    h = Histogram("llm_seconds", "demo", buckets=buckets, registry=reg)
    for x in latencies:
        h.observe(x)
    samples = [s for s in next(iter(reg.collect())).samples if s.name.endswith("_bucket")]
    uppers = [float(s.labels["le"]) for s in samples]; counts = [s.value for s in samples]
    rows.append({"buckets": name, "in +Inf": counts[-1] - counts[-2], "estimated p95 (s)": round(quantile_from_buckets(0.95, uppers, counts), 1)})
rows.append({"buckets": "true p95 of the data", "in +Inf": "", "estimated p95 (s)": float(np.percentile(latencies, 95))})
pd.DataFrame(rows)

,buckets,in +Inf,estimated p95 (s)
0,default (ends at 10 s),3.0,10.0
1,LLM buckets (to 120 s),0.0,56.0
2,true p95 of the data,,42.0


Default bucket-এ তিনটা latency `+Inf`-এ, আর p95 "১০ s" — যেখানে আসল p95 ৪০ s+। Dashboard সবুজ, user রাগী। LLM bucket-এর অনুমানও নিখুঁত না (৩২-৬৪ s bucket-এর ভেতরে linear interpolation), কিন্তু সঠিক পরিসরে — SLO threshold-এর কাছে বেশি bucket রাখলে অনুমান ধারালো হয়।

**Cardinality:** প্রতিটা আলাদা label মান = নতুন time series। নিচে route template বনাম প্রশ্নের text label-এ, ১,০০০টা request:

In [5]:
reg = CollectorRegistry()
by_route = Counter("by_route_requests", "demo", ["route"], registry=reg)
by_question = Counter("by_question_requests", "demo", ["question"], registry=reg)
for i in range(1000):
    by_route.labels("/v1/ask").inc()
    by_question.labels(f"How many sick days do I get? variant {i}").inc()
text = generate_latest(reg).decode()
print("series with route label   :", text.count("by_route_requests_total{"))
print("series with question label:", text.count("by_question_requests_total{"))
print("exposition size: ", len(text) // 1024, "KB for ONE metric — and every series lives in Prometheus memory")

series with route label   : 1
series with question label: 1000
exposition size:  183 KB for ONE metric — and every series lives in Prometheus memory


## 3. OpenTelemetry GenAI Span: `chat llama3.1:8b`

GenAI semantic conventions (এখনো **Development** status) বলে: inference span-এর নাম `{operation} {model}`, kind `CLIENT`; required `gen_ai.operation.name` আর `gen_ai.provider.name` (পুরোনো `gen_ai.system`-এর বদলি; "ollama" custom মান, চলে); recommended token usage আর `finish_reasons`। Prompt/answer content **opt-in**। নিচে একটা manual span আর in-memory exporter — কোনো backend লাগে না; production-এ একই span OTLP দিয়ে Phoenix/Langfuse-এ।

In [6]:
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider(resource=Resource.create({"service.name": "rag-api"}))
provider.add_span_processor(SimpleSpanProcessor(exporter))
tracer = provider.get_tracer("lecture")
CAPTURE_CONTENT = False                                     # opt-in: prompts carry PII

with tracer.start_as_current_span("rag.ask") as root:
    with tracer.start_as_current_span("retrieval", attributes={"gen_ai.operation.name": "retrieval"}):
        time.sleep(0.002)
    with tracer.start_as_current_span(f"chat {MODEL}", kind=trace.SpanKind.CLIENT) as s:
        s.set_attributes({"gen_ai.operation.name": "chat", "gen_ai.provider.name": "ollama",
                          "gen_ai.request.model": MODEL, "server.address": "localhost", "server.port": 11434})
        r = client.chat(model=MODEL, options=OPTS, keep_alive="30m",
                        messages=[{"role": "user", "content": f"CONTEXT:\n{CONTEXT}\n\nHow many sick days? Cite."}])
        s.set_attributes({"gen_ai.usage.input_tokens": r.prompt_eval_count, "gen_ai.usage.output_tokens": r.eval_count,
                          "gen_ai.response.finish_reasons": [r.done_reason]})
        if CAPTURE_CONTENT:
            s.set_attribute("gen_ai.output.messages", json.dumps([{"role": "assistant", "content": r.message.content}]))

for sp in exporter.get_finished_spans():
    print(f"{sp.name:20} kind={sp.kind.name:8} {(sp.end_time - sp.start_time)/1e6:8.1f} ms  parent={'yes' if sp.parent else 'root'}")
chat_span = next(sp for sp in exporter.get_finished_spans() if sp.name.startswith("chat"))
dict(chat_span.attributes)

retrieval            kind=INTERNAL      2.5 ms  parent=yes
chat llama3.1:8b     kind=CLIENT      811.4 ms  parent=yes
rag.ask              kind=INTERNAL    814.1 ms  parent=root


{'gen_ai.operation.name': 'chat',
 'gen_ai.provider.name': 'ollama',
 'gen_ai.request.model': 'llama3.1:8b',
 'server.address': 'localhost',
 'server.port': 11434,
 'gen_ai.usage.input_tokens': 109,
 'gen_ai.usage.output_tokens': 16,
 'gen_ai.response.finish_reasons': ('stop',)}

**Metric নাম বনাম attribute নাম:** spec-এর repo-তে metric নাম বদলাচ্ছে (`gen_ai.client.operation.*` → `gen_ai.client.inference.*`), কিন্তু released Python package এখনো পুরোনো নাম। Attribute vocabulary (উপরের dict) স্থিতিশীল — সেটাই শেখার জিনিস; metric নাম এক module-এ রাখুন যাতে rename এক লাইনের কাজ হয়।

## 4. সস্তা Quality Check: কী ধরে, কী ধরে না

HTTP 200 মানেই ভালো উত্তর না। প্রতি request-এ microsecond-এ চালানো যায় এমন দুটো check: **citation valid** (`[n]` আছে, আর n ≤ passage সংখ্যা) আর **numbers supported** (উত্তরের প্রতিটা সংখ্যা passage-এ আছে)। নিচে একটা আসল উত্তর আর কয়েকটা ইচ্ছাকৃত ভুল:

In [7]:
CITE = re.compile(r"\[(\d+)\]")
NUM = re.compile(r"(?<![\w.])\d+(?:[.,]\d+)?(?![\w])")

def cheap_checks(answer, passages=PASSAGES):
    cited = {int(n) for n in CITE.findall(answer)}
    nums = set(NUM.findall(CITE.sub(" ", answer))) - set(NUM.findall(" ".join(passages)))
    return {"citation_valid": bool(cited) and all(1 <= n <= len(passages) for n in cited),
            "numbers_supported": not nums, "unsupported": ",".join(sorted(nums))}

real_answer, _, _ = ask("How many sick days do employees get?")
CASES = {
    "llama3.1:8b (real)": real_answer,
    "invented 18": "Employees get 18 paid sick days per year [1].",
    "invented 15": "Employees get 15 paid sick days per year [1].",
    "bad citation [7]": "Employees get 12 paid sick days per year [7].",
    "invented claim": "Employees get 12 paid sick days per year, and unused sick days can be cashed out [1].",
    "paraphrase": "You get twelve paid sick days each year [1].",
}
checks = pd.DataFrame([{"case": k, "answer": v[:70], **cheap_checks(v)} for k, v in CASES.items()])
checks

,case,answer,citation_valid,numbers_supported,unsupported
0,llama3.1:8b (real),12 [1],True,True,
1,invented 18,Employees get 18 paid sick days per year [1].,True,False,18
2,invented 15,Employees get 15 paid sick days per year [1].,True,True,
3,bad citation [7],Employees get 12 paid sick days per year [7].,False,True,
4,invented claim,"Employees get 12 paid sick days per year, and ...",True,True,
5,paraphrase,You get twelve paid sick days each year [1].,True,True,


"18" আর "[7]" ধরা পড়ে। কিন্তু **"15" পার পায়** — কারণ remote-work passage-এ "15:00" আছে; check শুধু দেখে সংখ্যাটা context-এ *কোথাও* আছে কিনা। আর **"cashed out" দাবি** সংখ্যা-citation দুটো check-ই পাস করে। সস্তা check দরকারি, কিন্তু যথেষ্ট না।

## 5. LLM Judge: দুই Design, একই Model

Judge = আরেকটা LLM call: "উত্তরের প্রতিটা দাবি কি passage-এ আছে?"। দুটো design একই llama3.1:8b-এ:

* **A — শুধু রায়:** `{"supported": bool}`
* **B — claim ধরে ধরে:** প্রতিটা claim, passage থেকে হুবহু `evidence`, তারপর `supported`; আর code-এ যাচাই যে "supported" claim-এর quote সত্যিই passage-এ আছে।

JSON field-এর ক্রম = model যে ক্রমে লেখে — B-তে evidence আগে, রায় পরে।

In [8]:
from pydantic import BaseModel

class VerdictA(BaseModel):
    supported: bool

class Claim(BaseModel):
    claim: str
    evidence: str
    supported: bool

class VerdictB(BaseModel):
    claims: list[Claim]

SYS_A = "You are a strict fact-checker. Is every factual claim in the ANSWER supported by the PASSAGES?"
SYS_B = ("You are a strict fact-checker. Split the ANSWER into its factual claims (ignore citation markers like [1]). "
         "For each claim, copy the exact sentence from the PASSAGES that supports it into `evidence` (empty string if "
         "none), then set `supported`: true only if the evidence states the same fact (a number must have the same "
         "value, e.g. twelve = 12).")

def norm(t):
    return re.sub(r"\W+", " ", t.lower()).strip()

def judge(answer, design):
    model_cls, system = (VerdictA, SYS_A) if design == "A" else (VerdictB, SYS_B)
    r = client.chat(model=MODEL, format=model_cls.model_json_schema(), options=OPTS, keep_alive="30m",
                    messages=[{"role": "system", "content": system},
                              {"role": "user", "content": f"PASSAGES:\n{CONTEXT}\n\nANSWER: {answer}"}])
    v = model_cls.model_validate_json(r.message.content)
    if design == "A":
        return v.supported, ""
    bad = [c.claim for c in v.claims if not c.supported or norm(c.evidence) not in norm(" ".join(PASSAGES))]
    return (bool(v.claims) and not bad), "; ".join(bad)

EXPECTED = {"llama3.1:8b (real)": True, "invented 18": False, "invented 15": False, "bad citation [7]": True,
            "invented claim": False, "paraphrase": True}       # [7] is a citation problem, not a factual one
rows = []
t0 = time.perf_counter()
for name, text in CASES.items():
    a, _ = judge(text, "A")
    b, why = judge(text, "B")
    rows.append({"case": name, "expected": EXPECTED[name], "judge A": a, "judge B": b, "B found": why[:60]})
judged = pd.DataFrame(rows)
print(f"{2 * len(CASES)} judge calls in {time.perf_counter() - t0:.0f}s | "
      f"A correct {(judged['judge A'] == judged.expected).sum()}/{len(judged)} | B correct {(judged['judge B'] == judged.expected).sum()}/{len(judged)}")
judged

12 judge calls in 29s | A correct 5/6 | B correct 6/6


,case,expected,judge A,judge B,B found
0,llama3.1:8b (real),True,False,True,
1,invented 18,False,False,False,Employees get 18 paid sick days per year
2,invented 15,False,False,False,Employees get 15 paid sick days per year
3,bad citation [7],True,True,True,
4,invented claim,False,False,False,unused sick days can be cashed out
5,paraphrase,True,True,True,


**পড়ার নিয়ম:** সংখ্যাগুলো নিজের output থেকে পড়ুন — judge-এর আচরণ prompt আর schema-র ছোট পরিবর্তনে বদলায়। Project বানানোর সময় একই model-এ তিনটা design মাপা হয়েছিল: শুধু রায় (নিজের সাথে অসঙ্গত উত্তর দিয়েছিল), "unsupported claim-এর তালিকা আগে" (০/১১ — সব claim-কেই তালিকায় দিয়েছিল), claim → evidence → রায় + quote check (১১/১১)।

⚠️ **Self-preference:** এখানে judge আর উত্তরদাতা একই model — নিজের লেখা পছন্দ করার পক্ষপাত এড়ানো যায় না। Judge-এর score trend হিসেবে পড়ুন, আর ৫০-২০০টা উত্তর নিজে label করে judge-এর সাথে মেলান। Judge চলে **offline**, request-এর বাইরে।

## 6. SLO Burn Rate: কখন Page, কখন Ticket

SLO 99% → error budget 1%। **Burn rate** = error ratio ÷ 0.01। SRE Workbook-এর multiwindow নিয়ম: 1h **আর** 5m দুটোতেই > 14.4 হলে page; 6h আর 30m-এ > 6 হলে page; 3d আর 6h-এ > 1 হলে ticket। নিচে তিনটা পরিস্থিতি — প্রতি মিনিটে ১০টা request, ৬ ঘণ্টা:

In [9]:
def burn_rates(errors_per_min, req_per_min=10, budget=0.01):
    # errors_per_min: list of 360 minute buckets (oldest first); windows look back from "now"
    out = {}
    for label, minutes in {"5m": 5, "30m": 30, "1h": 60, "6h": 360}.items():
        e = sum(errors_per_min[-minutes:]); n = req_per_min * min(minutes, len(errors_per_min))
        out[label] = round(e / n / budget, 1)
    alerts = []
    if out["1h"] > 14.4 and out["5m"] > 14.4: alerts.append("PAGE fast burn")
    if out["6h"] > 6 and out["30m"] > 6: alerts.append("PAGE slow burn")
    return out | {"alerts": ", ".join(alerts) or "none"}

rng = np.random.default_rng(42)
scenarios = {
    "healthy (0.5% errors)": list(rng.binomial(10, 0.005, 360)),
    "slow leak (8% for 6h)": list(rng.binomial(10, 0.08, 360)),
    "outage last 20 min (60%)": [0] * 340 + list(rng.binomial(10, 0.6, 20)),
    "outage fixed 10 min ago": [0] * 320 + list(rng.binomial(10, 0.6, 30)) + [0] * 10,
}
pd.DataFrame({k: burn_rates(v) for k, v in scenarios.items()}).T

,5m,30m,1h,6h,alerts
healthy (0.5% errors),0.0,0.3,0.2,0.4,none
slow leak (8% for 6h),8.0,9.7,8.8,8.5,PAGE slow burn
outage last 20 min (60%),62.0,45.3,22.7,3.8,PAGE fast burn
outage fixed 10 min ago,0.0,45.3,33.7,5.6,none


**পড়ার নিয়ম:**

* **Outage চলছে** → 1h আর 5m দুটোই উঁচু → fast-burn page।
* **Outage ১০ মিনিট আগে ঠিক হয়েছে** → 1h এখনো উঁচু, কিন্তু 5m শূন্য → page **না**। ছোট window-এর কাজ এটাই: সমস্যা থামলে alert দ্রুত থামে।
* **Slow leak** → fast-burn চুপ, কিন্তু 6h/30m ধরে।

Project এই একই হিসাব দুই জায়গায় করে: `/v1/stats`-এ (SQLite থেকে) আর `monitoring/alerts.yml`-এ (Prometheus rule, `promtool test rules` দিয়ে test করা)। Quality signal (judge, not-found) কখনো page না — sample ছোট, noise বেশি → ticket।

## 7. Drift: KS আর Wasserstein

Drift = কোনো derived feature-এর বণ্টন বদলানো (প্রশ্নের দৈর্ঘ্য, top-1 distance, উত্তরের দৈর্ঘ্য, refusal rate)। Evidently-র default: ১০০০ row পর্যন্ত KS test (p < 0.05), বেশি হলে Wasserstein distance ≥ 0.1 (normalized)। নিচে প্রশ্নের দৈর্ঘ্য — baseline সপ্তাহ বনাম তিনটা নতুন সপ্তাহ:

In [10]:
from scipy.stats import ks_2samp, wasserstein_distance

rng = np.random.default_rng(0)
baseline = rng.normal(45, 12, 600).clip(5)
weeks = {"same users": rng.normal(45, 12, 600).clip(5),
         "slightly longer (+3 chars)": rng.normal(48, 12, 600).clip(5),
         "new user group (long, pasted text)": rng.normal(90, 30, 600).clip(5)}
rows = []
for name, wk in weeks.items():
    ks = ks_2samp(baseline, wk)
    w = wasserstein_distance(baseline, wk) / baseline.std()          # normalised so 0.1 is comparable
    rows.append({"week": name, "mean_len": round(wk.mean(), 1), "KS p-value": f"{ks.pvalue:.3g}",
                 "KS drift (p<0.05)": ks.pvalue < 0.05, "Wasserstein (norm.)": round(w, 2), "W drift (>=0.1)": w >= 0.1})
pd.DataFrame(rows)

,week,mean_len,KS p-value,KS drift (p<0.05),Wasserstein (norm.),W drift (>=0.1)
0,same users,44.5,0.675,False,0.05,False
1,slightly longer (+3 chars),48.0,1.14e-05,True,0.27,True
2,"new user group (long, pasted text)",89.0,1.91e-163,True,3.72,True


**পড়ার নিয়ম:**

* **KS** গড় প্রশ্নে মাত্র ৩ অক্ষরের পরিবর্তনকে প্রায় শূন্য p-value দিয়ে "drift" বলে — p-value বলে পার্থক্য *আছে*, অর্থপূর্ণ কিনা বলে না; sample যত বড়, তত ছোট পার্থক্যও significant।
* **Wasserstein** effect size: কতটা সরেছে। কিন্তু default 0.1 কি আমাদের data-র noise-এর উপরে? সেটা মাপতে হয় — নিচে ২০টা "কিছুই বদলায়নি" সপ্তাহ simulate করে:

In [11]:
normal_weeks = [wasserstein_distance(baseline, rng.normal(45, 12, 600).clip(5)) / baseline.std() for _ in range(20)]
threshold = round(float(np.percentile(normal_weeks, 95)) * 1.5, 2)          # noise ceiling + margin
print(f"W on 20 unchanged weeks: median {np.median(normal_weeks):.3f}, max {max(normal_weeks):.3f}; "
      f"{sum(w >= 0.1 for w in normal_weeks)}/20 would cross the default 0.1 -> calibrated threshold {threshold}")
pd.DataFrame([{"week": r["week"], "Wasserstein (norm.)": r["Wasserstein (norm.)"],
               f"drift at calibrated {threshold}": r["Wasserstein (norm.)"] >= threshold} for r in rows])

W on 20 unchanged weeks: median 0.069, max 0.126; 2/20 would cross the default 0.1 -> calibrated threshold 0.19


,week,Wasserstein (norm.),drift at calibrated 0.19
0,same users,0.05,False
1,slightly longer (+3 chars),0.27,True
2,"new user group (long, pasted text)",3.72,True


কিছু না বদলালেও কয়েকটা সপ্তাহ default 0.1 পার করে — প্রতি সপ্তাহে এক-দুটো মিথ্যা alarm মানে কিছুদিন পর কেউ আর drift report পড়ে না। Noise থেকে বানানো threshold-এ "same users" চুপ, নতুন user group স্পষ্ট লাল (noise-এর কয়েক ডজন গুণ), আর +৩ অক্ষরও লাল — সেটা আসল কিন্তু ছোট পরিবর্তন; অর্থপূর্ণ কিনা product-এর প্রশ্ন, statistics-এর না। Drift তাই dashboard বা সাপ্তাহিক report-এ, pager-এ না।

## 📝 Key Takeaways

* **Ollama নিজেই telemetry দেয়:** token, prefill, decode, load — decode tok/s, TTFT, GPU-seconds, cold start সব হিসাব করা যায়।
* **Prompt-prefix cache TTFT-কে bimodal করে** — percentile দেখুন, load test-এ প্রশ্ন বদলান।
* **Bucket LLM-আকারের** (default 10 s-এ শেষ → p95 মিথ্যা); **label-এ কখনো user data না**।
* **GenAI span:** `chat llama3.1:8b`, CLIENT, `gen_ai.*` attribute; content opt-in; metric নাম এখনো বদলাচ্ছে।
* **Quality স্তরে:** সস্তা check সবকিছুতে (কিন্তু "15" আর বানানো দাবি পার পায়); judge offline, claim → evidence → রায়, আর judge-কেও যাচাই।
* **Page শুধু multiwindow burn-এ;** quality আর drift → ticket/report।

## 🧠 Brain Teasers

1. Section 2-এ latencies-এ একটা `200.0` যোগ করুন। LLM bucket-এ p95 কত হয়? Bucket কোথায় শেষ হওয়া উচিত?
2. Section 3-এ `CAPTURE_CONTENT = True` করুন। Span-এ কী যোগ হয়? এটা production-এ কেন বিপজ্জনক?
3. Section 4-এর number check-কে এমনভাবে বদলান যাতে "15" ধরা পড়ে (hint: সংখ্যার পাশের শব্দ — "15 sick days" বনাম "15:00")। কোন নতুন false positive তৈরি হয়?
4. Section 5-এর `SYS_B`-তে "e.g. twelve = 12" অংশটা মুছে দিন। "paraphrase" case-এর রায় কী হয়?
5. Section 6-এ প্রতি মিনিটে ১০টার বদলে ঘণ্টায় ১০টা request আর একটা error ধরুন। 1h burn rate কত? Low traffic-এ SRE Workbook কী পরামর্শ দেয়?